# 01 — EDA y refinamiento (Spark)

**Fase 3 del obligatorio.** Lee desde `/movilidad/raw/`, aplica limpieza y guarda Parquet en `/movilidad/refined/`.

> Solo Spark en esta etapa (sin pandas). Ver reglas en `plan/04-limpieza-y-refinamiento.md`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

spark = (
    SparkSession.builder
    .appName("movilidad-eda-refinamiento")
    .getOrCreate()
)

RAW = "hdfs:///movilidad/raw"
REFINED = "hdfs:///movilidad/refined"

spark.sparkContext.setLogLevel("WARN")

In [ ]:
def perfil_tabla(df, nombre):
    """EDA estándar exigido por la letra."""
    print(f"\n{'='*60}")
    print(f"TABLA: {nombre}")
    print(f"{'='*60}")
    print(f"Filas: {df.count():,}  |  Columnas: {len(df.columns)}")
    print("\nEsquema:")
    df.printSchema()
    print("\nVista previa:")
    df.show(5, truncate=50)
    print("\nNulos por columna:")
    exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    df.agg(*exprs).show(truncate=False)


def guardar_refined(df, nombre):
    path = f"{REFINED}/{nombre}"
    df.write.mode("overwrite").parquet(path)
    print(f"Guardado: {path}  ({df.count():,} filas)")

## 1. Catálogo — `tipos_vehiculos_dag`

In [ ]:
tipos_raw = spark.read.option("header", True).option("sep", ";").csv(f"{RAW}/tipos_vehiculos_dag.csv")
perfil_tabla(tipos_raw, "tipos_vehiculos_dag (raw)")

tipos = (
    tipos_raw
    .withColumnRenamed("cod_tipveh", "cod_tipveh")
    .withColumn("cod_tipveh", F.col("cod_tipveh").cast(IntegerType()))
    .withColumn("tipo_vehiculo_limpio", F.regexp_replace(F.col("tipo_vehiculo"), r"^[._\s]+", ""))
    .withColumn(
        "es_tipo_valido",
        ~F.lower(F.col("tipo_vehiculo")).rlike("invalido|inva")
    )
    .dropDuplicates(["cod_tipveh"])
)

guardar_refined(tipos, "tipos_vehiculos")

## 2. Parque automotor — 2019 y 2024

Reglas: RB-P1 a RB-P5. Join con tipos. Normalización de combustible.

In [ ]:
def _es_col_anio(col_name):
    norm = col_name.lower().strip().strip('"').replace("ñ", "n").replace("\ufffd", "")
    return norm in ("ano", "anio", "año", "ao") or (norm.startswith("a") and norm.endswith("o") and len(norm) <= 4)


def leer_parque(path, anio_referencia):
    df = (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")  # 2024 trae 'año' en latin-1; 2019 trae 'anio'
        .csv(path)
    )
    for col_name in df.columns:
        if _es_col_anio(col_name):
            df = df.withColumnRenamed(col_name, "anio")
            break
    return (
        df
        .withColumn("anio", F.col("anio").cast(IntegerType()))
        .withColumn("cod_tipveh", F.regexp_replace("cod_tipveh", '"', "").cast(IntegerType()))
        .withColumn("cantidad", F.col("cantidad").cast(IntegerType()))
        .withColumn("combustible", F.upper(F.trim(F.col("combustible"))))
        .withColumn(
            "combustible_norm",
            F.when(F.col("combustible").isin("GASOIL"), "DIESEL")
             .when(F.col("combustible").isin("OTRO"), "OTROS")
             .otherwise(F.col("combustible"))
        )
        .withColumn(
            "categoria_combustible",
            F.when(F.col("combustible_norm").isin("NAFTA", "DIESEL"), "TRADICIONAL")
             .when(F.col("combustible_norm") == "ELECTRICO", "ELECTRICO")
             .when(F.col("combustible_norm") == "HIBRIDO", "HIBRIDO")
             .when(F.col("combustible_norm") == "SIN MOTOR", "SIN_MOTOR")
             .otherwise("OTROS")
        )
        .withColumn("anio_referencia", F.lit(anio_referencia))
        .filter(F.col("cantidad") > 0)
        .filter(F.col("anio").between(1900, anio_referencia))
    )


parque_2019_raw = leer_parque(f"{RAW}/parque_dag_2019.csv", 2019)
parque_2024_raw = leer_parque(f"{RAW}/parque_dag_2024.csv", 2024)

perfil_tabla(parque_2019_raw, "parque_dag_2019 (raw)")
perfil_tabla(parque_2024_raw, "parque_dag_2024 (raw)")

def refinar_parque(df):
    return (
        df.join(tipos.select("cod_tipveh", "tipo_vehiculo_limpio", "es_tipo_valido"), "cod_tipveh", "left")
        .groupBy(
            "marca", "modelo", "anio", "combustible_norm", "cod_tipveh",
            "anio_referencia", "categoria_combustible", "tipo_vehiculo_limpio", "es_tipo_valido"
        )
        .agg(F.sum("cantidad").alias("cantidad"))
    )

parque_2019 = refinar_parque(parque_2019_raw)
parque_2024 = refinar_parque(parque_2024_raw)

print(f"Total parque 2019: {parque_2019.agg(F.sum('cantidad')).collect()[0][0]:,}")
print(f"Total parque 2024: {parque_2024.agg(F.sum('cantidad')).collect()[0][0]:,}")

guardar_refined(parque_2019, "parque_2019")
guardar_refined(parque_2024, "parque_2024")

## 3. Sensores — metadata de ubicación

In [ ]:
def leer_sensores(path, anio_referencia, sep):
    return (
        spark.read.option("header", True).option("sep", sep).csv(path)
        .withColumn("latitud", F.col("latitud").cast("double"))
        .withColumn("longitud", F.col("longitud").cast("double"))
        .withColumn("anio_referencia", F.lit(anio_referencia))
    )

sensores_2019 = leer_sensores(f"{RAW}/infosensoresmedicionconteo012021.csv", 2019, ";")
sensores_2024 = leer_sensores(f"{RAW}/infosensoresmedicionconteo012024.csv", 2024, ",")

perfil_tabla(sensores_2019, "sensores 2019-proxy (raw)")
perfil_tabla(sensores_2024, "sensores 2024 (raw)")

guardar_refined(sensores_2019, "sensores_2019")
guardar_refined(sensores_2024, "sensores_2024")

## 4. `dim_ubicacion` — clave de integración entre tablas

Se genera desde sensores + conteo + velocidad. **No** unir años por `id_detector`.

In [ ]:
def clave_ubicacion(df):
    return df.select(
        "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"
    ).distinct()


# Ubicaciones desde sensores (luego se enriquece con conteo/velocidad en celdas siguientes)
ubicaciones = clave_ubicacion(sensores_2019).unionByName(clave_ubicacion(sensores_2024)).distinct()

dim_ubicacion = (
    ubicaciones
    .withColumn(
        "id_ubicacion",
        F.abs(F.hash("dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"))
    )
    .withColumn(
        "tramo_descripcion",
        F.concat_ws(" | ", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente")
    )
)

print(f"Ubicaciones únicas (sensores): {dim_ubicacion.count():,}")
guardar_refined(dim_ubicacion, "dim_ubicacion")

## 5. Conteo vehicular — 2019 y 2024

In [ ]:
def leer_conteo(path, anio_referencia):
    df = spark.read.option("header", True).option("sep", ",").csv(path)
    # Unificar nombres de columnas entre años
    for old, new in [("cod_detector", "id_detector"), ("volume", "volumen")]:
        if old in df.columns:
            df = df.withColumnRenamed(old, new)
    return (
        df
        .withColumn("id_detector", F.col("id_detector").cast(IntegerType()))
        .withColumn("id_carril", F.col("id_carril").cast(IntegerType()))
        .withColumn("volumen", F.col("volumen").cast(IntegerType()))
        .withColumn("volumen_hora", F.col("volumen_hora").cast("double"))
        .withColumn("latitud", F.col("latitud").cast("double"))
        .withColumn("longitud", F.col("longitud").cast("double"))
        .withColumn("hora", F.regexp_replace(F.col("hora").cast("string"), r"\.\d+$", ""))
        .withColumn("anio_referencia", F.lit(anio_referencia))
        .filter(F.col("volumen") >= 0)
        .filter(F.col("dsc_avenida").isNotNull())
        .filter(F.col("latitud").between(-34.95, -34.78))
        .filter(F.col("longitud").between(-56.35, -56.05))
    )


conteo_2019_raw = leer_conteo(f"{RAW}/autoscope_ene_2019.csv", 2019)
conteo_2024_raw = leer_conteo(f"{RAW}/autoscope_01_2024_volumen.csv", 2024)

perfil_tabla(conteo_2019_raw, "conteo 2019 (raw)")
print(f"Detectores únicos 2019: {conteo_2019_raw.select('id_detector').distinct().count()}")
perfil_tabla(conteo_2024_raw, "conteo 2024 (raw)")
print(f"Detectores únicos 2024: {conteo_2024_raw.select('id_detector').distinct().count()}")

def refinar_conteo(df):
    clave = ["id_detector", "id_carril", "fecha", "hora", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente"]
    return (
        df.groupBy(*clave, "latitud", "longitud", "anio_referencia")
        .agg(
            F.sum("volumen").alias("volumen"),
            F.avg("volumen_hora").alias("volumen_hora"),
        )
        .join(dim_ubicacion.select("id_ubicacion", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"),
              ["dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"], "left")
    )

conteo_2019 = refinar_conteo(conteo_2019_raw)
conteo_2024 = refinar_conteo(conteo_2024_raw)

guardar_refined(conteo_2019, "conteo_2019")
guardar_refined(conteo_2024, "conteo_2024")

## 6. Velocidad — 2019-proxy (2021) y 2024

In [ ]:
def leer_velocidad(path, anio_referencia, sep):
    df = spark.read.option("header", True).option("sep", sep).option("encoding", "UTF-8").csv(path)
    for old, new in [("cod_detector", "id_detector"), ("velocidad", "velocidad_promedio")]:
        if old in df.columns:
            df = df.withColumnRenamed(old, new)
    # BOM en header 2021
    if "\ufeffcod_detector" in df.columns or any(c.startswith("\ufeff") for c in df.columns):
        df = df.toDF(*[c.replace("\ufeff", "") for c in df.columns])
        if "cod_detector" in df.columns:
            df = df.withColumnRenamed("cod_detector", "id_detector")
    return (
        df
        .withColumn("id_detector", F.col("id_detector").cast(IntegerType()))
        .withColumn("id_carril", F.col("id_carril").cast(IntegerType()))
        .withColumn("velocidad_promedio", F.col("velocidad_promedio").cast(IntegerType()))
        .withColumn("latitud", F.col("latitud").cast("double"))
        .withColumn("longitud", F.col("longitud").cast("double"))
        .withColumn("hora", F.regexp_replace(F.col("hora").cast("string"), r"\.\d+$", ""))
        .withColumn("anio_referencia", F.lit(anio_referencia))
        .filter(F.col("velocidad_promedio").between(0, 149))
        .filter(F.col("latitud").between(-34.95, -34.78))
        .filter(F.col("longitud").between(-56.35, -56.05))
    )


vel_2019_raw = leer_velocidad(f"{RAW}/autoscope_01_2021_velocidad.csv", 2019, ";")
vel_2024_raw = leer_velocidad(f"{RAW}/autoscope_01_2024_velocidad.csv", 2024, ",")

perfil_tabla(vel_2019_raw, "velocidad 2019-proxy (raw)")
perfil_tabla(vel_2024_raw, "velocidad 2024 (raw)")

def refinar_velocidad(df):
    clave = ["id_detector", "id_carril", "fecha", "hora", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente"]
    return (
        df.groupBy(*clave, "latitud", "longitud", "anio_referencia")
        .agg(F.avg("velocidad_promedio").alias("velocidad_promedio"))
        .join(dim_ubicacion.select("id_ubicacion", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"),
              ["dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"], "left")
    )

velocidad_2019 = refinar_velocidad(vel_2019_raw)
velocidad_2024 = refinar_velocidad(vel_2024_raw)

guardar_refined(velocidad_2019, "velocidad_2019")
guardar_refined(velocidad_2024, "velocidad_2024")

## 7. Reconstruir `dim_ubicacion` completa y re-asignar `id_ubicacion`

Incluye tramos de conteo y velocidad que no estaban en metadata de sensores.

In [ ]:
ubicaciones_all = (
    clave_ubicacion(sensores_2019)
    .unionByName(clave_ubicacion(sensores_2024))
    .unionByName(clave_ubicacion(conteo_2019_raw))
    .unionByName(clave_ubicacion(conteo_2024_raw))
    .unionByName(clave_ubicacion(vel_2019_raw))
    .unionByName(clave_ubicacion(vel_2024_raw))
    .distinct()
)

dim_ubicacion = (
    ubicaciones_all
    .withColumn(
        "id_ubicacion",
        F.abs(F.hash("dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"))
    )
    .withColumn(
        "tramo_descripcion",
        F.concat_ws(" | ", "dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente")
    )
)

print(f"Ubicaciones únicas (todas las fuentes): {dim_ubicacion.count():,}")
guardar_refined(dim_ubicacion, "dim_ubicacion")

dim_cols = ["dsc_avenida", "dsc_int_anterior", "dsc_int_siguiente", "latitud", "longitud"]

def reasignar_ubicacion(df):
    base = df.drop("id_ubicacion") if "id_ubicacion" in df.columns else df
    return base.join(dim_ubicacion.select("id_ubicacion", *dim_cols), dim_cols, "left")

conteo_2019 = reasignar_ubicacion(conteo_2019)
conteo_2024 = reasignar_ubicacion(conteo_2024)
velocidad_2019 = reasignar_ubicacion(velocidad_2019)
velocidad_2024 = reasignar_ubicacion(velocidad_2024)

guardar_refined(conteo_2019, "conteo_2019")
guardar_refined(conteo_2024, "conteo_2024")
guardar_refined(velocidad_2019, "velocidad_2019")
guardar_refined(velocidad_2024, "velocidad_2024")

## 8. Validaciones post-refinamiento

In [ ]:
checks = {
    "parque_2019": (parque_2019, 625_522),
    "parque_2024": (parque_2024, 725_005),
}

for nombre, (df, esperado) in checks.items():
    total = df.agg(F.sum("cantidad")).collect()[0][0]
    diff_pct = abs(total - esperado) / esperado * 100
    status = "OK" if diff_pct < 2 else "REVISAR"
    print(f"{status}  {nombre}: total={total:,}  esperado≈{esperado:,}  diff={diff_pct:.1f}%")

# Cobertura id_ubicacion en conteo
for nombre, df in [("conteo_2019", conteo_2019), ("conteo_2024", conteo_2024)]:
    total = df.count()
    con_id = df.filter(F.col("id_ubicacion").isNotNull()).count()
    pct = con_id / total * 100 if total else 0
    print(f"Cobertura id_ubicacion {nombre}: {pct:.1f}% ({con_id:,}/{total:,})")

print("\nRefined en HDFS:")
!hdfs dfs -ls hdfs:///movilidad/refined